<a href="https://colab.research.google.com/github/Deepasivakumar25/AI_Learning/blob/main/cross_encodoing_re_ranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers accelerate numpy scikit-learn numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 57.1 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()

Saving Hybrid_Search_Practice.pdf to Hybrid_Search_Practice.pdf


In [3]:
from pypdf import PdfReader
reader = PdfReader("Hybrid_Search_Practice.pdf")
pdf_text = "\n".join(rec.extract_text() for rec in reader.pages)
print(pdf_text)

Hybrid Search Practice Document
Section 1: Artificial Intelligence
Artificial Intelligence (AI) enables computers to perform tasks that normally require human
intelligence. Machine Learning (ML) is a branch of AI that learns from data. Deep Learning is a
subset of ML that uses neural networks.
Section 2: Computer Vision
CNN (Convolutional Neural Network) is commonly used for image classification and object
detection. RNN (Recurrent Neural Network) is useful for sequential data such as text and speech.
Transformers are widely used in modern large language models.
Section 3: Retrieval-Augmented Generation (RAG)
A RAG system splits documents into chunks, converts them into embeddings, stores them in
FAISS, retrieves the most relevant chunks, and sends them to an LLM. Semantic search finds
information based on meaning, while keyword search finds exact words. Hybrid Search combines
semantic search with keyword search for better retrieval accuracy.
Section 4: Technical Terms
Version: Python 

In [4]:
chunk_size = 50
word_split = pdf_text.split()
chunk_list = [" ".join(word_split[i:i+chunk_size]) for i in range(0, len(word_split),chunk_size)]
print(chunk_list)

['Hybrid Search Practice Document Section 1: Artificial Intelligence Artificial Intelligence (AI) enables computers to perform tasks that normally require human intelligence. Machine Learning (ML) is a branch of AI that learns from data. Deep Learning is a subset of ML that uses neural networks. Section 2: Computer Vision CNN (Convolutional', 'Neural Network) is commonly used for image classification and object detection. RNN (Recurrent Neural Network) is useful for sequential data such as text and speech. Transformers are widely used in modern large language models. Section 3: Retrieval-Augmented Generation (RAG) A RAG system splits documents into chunks, converts them into embeddings,', 'stores them in FAISS, retrieves the most relevant chunks, and sends them to an LLM. Semantic search finds information based on meaning, while keyword search finds exact words. Hybrid Search combines semantic search with keyword search for better retrieval accuracy. Section 4: Technical Terms Version:

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

chatbot = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [36]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chunk_embedding = embedding_model.encode(chunk_list)
dimension = chunk_embedding.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embedding)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [37]:
question = "what is mean by hybrid search?"
question_embedding = embedding_model.encode([question])

distance, index_number = index.search(
    np.array(question_embedding),
    k=5
)

retrieved_chunks = [chunk_list[rec] for rec in index_number[0]]


In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
x = vectorizer.fit_transform(chunk_list)

question_tfidf = vectorizer.transform([question])

similarity_scores = cosine_similarity(question_tfidf,x)

top_k = 5

top_indices = np.argsort(similarity_scores[0])[::-1][:top_k]
keyword_chunk = [chunk_list[rec] for rec in top_indices]

keyword_indices = top_indices.tolist()
semantic_indices = index_number[0].tolist()

print(keyword_indices,semantic_indices)
combine = keyword_indices + semantic_indices

unique_list = list(dict.fromkeys(combine))
print(unique_list)

unique_list_text = [chunk_list[rec] for rec in unique_list]
print(unique_list_text,"________________")

context_list = [[question,chunk_list[rec]] for rec in unique_list]
print(context_list)



[2, 0, 1, 3] [2, 0, 1, 3, -1]
[2, 0, 1, 3, -1]
['stores them in FAISS, retrieves the most relevant chunks, and sends them to an LLM. Semantic search finds information based on meaning, while keyword search finds exact words. Hybrid Search combines semantic search with keyword search for better retrieval accuracy. Section 4: Technical Terms Version: Python 3.12 Library: FAISS Embedding', 'Hybrid Search Practice Document Section 1: Artificial Intelligence Artificial Intelligence (AI) enables computers to perform tasks that normally require human intelligence. Machine Learning (ML) is a branch of AI that learns from data. Deep Learning is a subset of ML that uses neural networks. Section 2: Computer Vision CNN (Convolutional', 'Neural Network) is commonly used for image classification and object detection. RNN (Recurrent Neural Network) is useful for sequential data such as text and speech. Transformers are widely used in modern large language models. Section 3: Retrieval-Augmented Genera

In [35]:
from sentence_transformers import CrossEncoder

# Load a pre-trained CrossEncoder model
cross_encoder_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Predict scores for a pair of sentences
scores = model.predict(context_list)
print(scores)
top_score_indices = np.argsort(scores)[::-1]
print(top_score_indices)

best_cross_encoders_chunks = []

for idx in top_score_indices[:3]:
    best_cross_encoders_chunks.append(unique_list_text[idx])

context = "\n\n".join(best_cross_encoders_chunks)

print(context)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[  4.8372035   3.778216   -9.995053  -11.449215  -11.449215 ]
[0 1 2 4 3]
stores them in FAISS, retrieves the most relevant chunks, and sends them to an LLM. Semantic search finds information based on meaning, while keyword search finds exact words. Hybrid Search combines semantic search with keyword search for better retrieval accuracy. Section 4: Technical Terms Version: Python 3.12 Library: FAISS Embedding

Hybrid Search Practice Document Section 1: Artificial Intelligence Artificial Intelligence (AI) enables computers to perform tasks that normally require human intelligence. Machine Learning (ML) is a branch of AI that learns from data. Deep Learning is a subset of ML that uses neural networks. Section 2: Computer Vision CNN (Convolutional

Neural Network) is commonly used for image classification and object detection. RNN (Recurrent Neural Network) is useful for sequential data such as text and speech. Transformers are widely used in modern large language models. Section 3: Retri

In [33]:
prompt = f"""
<|user|>

Use ONLY the context below.

Context:
{context}

Question:
{question}

If the answer is not present, reply exactly:

I couldn't find that information.

<|assistant|>
"""
response = chatbot(
    prompt,
    max_new_tokens=120,
    do_sample=False,
    return_full_text=False
)

answer = response[0]["generated_text"].strip()

print(answer)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Hybrid Search combines semantic search with keyword search for better retrieval accuracy.
